# 05 - Pipeline nowcast (OPTIONAL, runs after 03; not in the weekly job until the gate passes)

In-quarter **nowcast** of Growth ARR landing for UCC and ITSG:
`nowcast = quarter-to-date actual + sum over open opportunities of amount x scale x P(win in this quarter)`.

## NO-LEAKAGE RULES (read before editing)
1. **Features are point-in-time.** For an as-of Monday 00:00, each opportunity's stage / amount / close date is its
   latest `OpportunityHistory` row with change timestamp **strictly before** that Monday. Nothing later is visible.
2. **Labels may use the future, features never.** Whether a deal eventually landed in the quarter is a label only.
3. **Expanding window.** Win rates and the amount scale used to score a quarter are learned only from snapshots whose
   quarter ended **before** the as-of date. Baseline parameters and blend weights likewise use prior quarters only.
4. `cds_sfdc_opp_products_latest` is current-state: used only for the product / Growth / demo_stage classification
   and for the actual (closed-won) landing, never for stage.
5. `gaim_pipeline_daily_snapshot` is a single snapshot day and is NOT used.

## Gate
`validated = True` only if nowcast MAPE < best-baseline MAPE at as-of weeks >= 5 for BOTH UCC and ITSG
(baselines: seasonal-naive x growth, QTD pace + learned quarter-end uplift, blend). Until then outputs carry
`validated = False` and the app must not present them as a forecast.

## How to run
1. Run the discovery cell; set `HISTORY_TABLE` / `COLUMN_MAP` if the defaults are wrong (the table is UNVERIFIED).
2. Run top to bottom. The self-test cell needs no Spark and runs before any Spark cell.
3. Writes only `pipeline_nowcast` and `pipeline_nowcast_backtest`; no existing table is touched.

In [ ]:
# CONFIG (pure python; no Spark)
import numpy as np
import pandas as pd
from datetime import date

CATALOG = "datagroup_mdl"
SCHEMA = "mdl_sales_analytics"
APP_SERVICE_PRINCIPAL = "324a6ec7-e988-42c7-8a7f-55465f5bea37"

# Current-state opp-products table: classification + actuals ONLY (never stage).
BASE_TABLE = "datagroup.datalake_transform.cds_sfdc_opp_products_latest"
# UNVERIFIED location of the OpportunityHistory dump: override if discovery finds it elsewhere.
HISTORY_TABLE = "datagroup.datalake_transform.goto_sfdc_opportunity_history"
START_DATE = "2023-01-01"          # same lower bound as notebooks 01/02 for actuals

# Logical name -> candidate physical column names (matched case-insensitively, first hit wins).
COLUMN_MAP = {
    "id": ["id"],
    "opp_id": ["opportunityid", "opportunity_id"],
    "stage": ["stagename", "stage_name"],
    "forecast_cat": ["forecastcategory", "forecastcategoryname", "forecast_category"],
    "amount": ["amount"],
    "close_date": ["closedate", "close_date"],
    "probability": ["probability"],
    "change_ts": ["createddate", "created_date", "systemmodstamp"],
    "is_deleted": ["isdeleted", "is_deleted"],
    "data_day": ["data_day"],
}
REQUIRED_LOGICAL = ["id", "opp_id", "stage", "amount", "close_date", "change_ts", "data_day"]

PRODUCTS = ["UCC", "ITSG"]
ITSG_EXCLUDED_CHANNELS = ("Care", "Sales Other")   # as notebook 02 (UCC has no channel filter)

ASOF_WEEKS = [2, 5, 8, 11]          # as-of = quarter start Monday + (w-1) weeks => w-th week of quarter
WEEK_BUCKET_EDGES = [0, 2, 4, 6, 8, 10, 99]   # weeks-to-quarter-end bucket edges (right-open)
MIN_TRAIN_QUARTERS = 4              # prior quarters needed before a quarter is scored
SHRINK_K_CELL = 30.0                # pseudo-count: (stage x bucket x product) cell shrinks toward stage rate
SHRINK_K_STAGE = 30.0               # pseudo-count: stage rate shrinks toward global rate
MIN_AMOUNT_SCALE, MAX_AMOUNT_SCALE = 0.05, 5.0   # sanity clamp on amount -> Growth ARR scale
GATE_MIN_ASOF_WEEK = 5

PIPELINE_TABLE = f"{CATALOG}.{SCHEMA}.pipeline_nowcast"
BACKTEST_TABLE = f"{CATALOG}.{SCHEMA}.pipeline_nowcast_backtest"
RUN_SELF_TEST = True

In [ ]:
# CORE PART 1 (pure python): stage mapping, quarter maths, point-in-time state.
import re

OPEN_STAGE_KEYS = ["discovery", "qualify", "evaluate", "negotiate", "pipeline", "bestcase", "commit", "other_open"]


def stage_key(stage, forecast_cat=None):
    """Map raw stage (+ forecast category) to a stable key. Won/lost take precedence; unknown -> 'other_open'."""
    s = re.sub(r"^\s*\d+\s*[.\-)]\s*", "", str(stage or "")).strip().lower()
    f = str(forecast_cat or "").strip().lower()
    if "lost" in s or s == "closed lost":
        return "lost"
    if "won" in s:
        return "won"
    if f == "closed" and not s:
        return "won_or_lost_unknown"
    for k in ("discovery", "qualify", "evaluate", "negotiate"):
        if k in s:
            return k
    if f in ("pipeline", "best case", "bestcase", "commit"):
        return f.replace(" ", "")
    return "other_open"


def is_open_key(k):
    return k in OPEN_STAGE_KEYS


def quarter_start(ts):
    """Calendar-quarter start (Timestamp midnight) containing ts."""
    ts = pd.Timestamp(ts).normalize()
    return pd.Timestamp(year=ts.year, month=3 * ((ts.month - 1) // 3) + 1, day=1)


def next_quarter_start(ts):
    """Exclusive end of the quarter containing ts (handles the Q4 -> Q1 year boundary)."""
    return quarter_start(ts) + pd.DateOffset(months=3)


def quarter_label(ts):
    q = quarter_start(ts)
    return f"{q.year}Q{(q.month - 1) // 3 + 1}"


def week_monday(ts):
    ts = pd.Timestamp(ts).normalize()
    return ts - pd.Timedelta(days=ts.weekday())


def first_monday_on_or_after(ts):
    ts = pd.Timestamp(ts).normalize()
    return ts + pd.Timedelta(days=(7 - ts.weekday()) % 7)


def asof_monday_for(qstart, week_no):
    """As-of Monday for the week_no-th week of the quarter: first Monday on/after quarter start + (week_no-1) weeks."""
    return first_monday_on_or_after(qstart) + pd.Timedelta(weeks=week_no - 1)


def weeks_to_qend(asof):
    """Whole weeks from as-of to the exclusive quarter end (floor)."""
    return int((next_quarter_start(asof) - pd.Timestamp(asof).normalize()).days // 7)


def weeks_bucket(w, edges=None):
    """Bucket index of weeks-to-quarter-end using right-open edges, e.g. [0,2) -> 0."""
    edges = edges or WEEK_BUCKET_EDGES
    for i in range(len(edges) - 1):
        if edges[i] <= w < edges[i + 1]:
            return i
    return len(edges) - 2


def asof_state(history, asof_ts):
    """Latest history row per opportunity with change_ts STRICTLY BEFORE asof_ts.

    history: columns opp_id, change_ts, id (tie-break), stage, forecast_cat, amount, close_date.
    Rows at/after asof_ts are dropped before anything else, so no future information can leak.
    """
    asof_ts = pd.Timestamp(asof_ts)
    h = history[pd.to_datetime(history["change_ts"]) < asof_ts]
    h = h.sort_values(["opp_id", "change_ts", "id"], kind="mergesort")
    return h.groupby("opp_id", as_index=False).tail(1).reset_index(drop=True)

In [ ]:
# CORE PART 2 (pure python): win rates, amount scale, nowcast arithmetic.
# `snaps` = point-in-time rows: asof, opp_id, product, stage, forecast_cat, amount, close_date
# `finals` = FINAL state per opp (LABEL source only): opp_id, final_stage, final_forecast_cat, final_close_date
# `arr_df` = realised Growth ARR per opp: opp_id, product, arr   (sum of amount_towards_plan, closed-won)


def prepare_snaps(snaps):
    """Add key / quarter / bucket columns. Uses only the snapshot's own columns (no future info)."""
    s = snaps.copy()
    s["asof"] = pd.to_datetime(s["asof"])
    s["close_date"] = pd.to_datetime(s["close_date"])
    s["amount"] = pd.to_numeric(s["amount"], errors="coerce").fillna(0.0)
    s["key"] = [stage_key(a, b) for a, b in zip(s["stage"], s["forecast_cat"])]
    s["qstart"] = s["asof"].map(quarter_start)
    s["qend"] = s["asof"].map(next_quarter_start)
    s["qlabel"] = s["asof"].map(quarter_label)
    s["bucket"] = s["asof"].map(lambda a: weeks_bucket(weeks_to_qend(a)))
    s["is_open"] = s["key"].map(is_open_key)
    s["in_q"] = (s["close_date"] >= s["qstart"]) & (s["close_date"] < s["qend"])
    return s


def open_in_quarter(prepared):
    """Universe scored/trained on: open at as-of AND close date inside the as-of quarter."""
    return prepared[prepared["is_open"] & prepared["in_q"]].copy()


def add_labels(open_rows, finals):
    """LABEL (may use the future): did the opp end Won with a final close date inside the snapshot's quarter?"""
    f = finals.copy()
    f["fkey"] = [stage_key(a, b) for a, b in zip(f["final_stage"], f["final_forecast_cat"])]
    f["final_close_date"] = pd.to_datetime(f["final_close_date"])
    m = open_rows.merge(f[["opp_id", "fkey", "final_close_date"]], on="opp_id", how="left")
    m["label"] = ((m["fkey"] == "won") & (m["final_close_date"] >= m["qstart"])
                  & (m["final_close_date"] < m["qend"])).astype(int)
    return m


def win_rates(labeled, asof, min_train_quarters=None, k_cell=None, k_stage=None):
    """P(win in-quarter | stage, weeks-bucket, product), hierarchical shrinkage cell -> stage -> global.

    Trains ONLY on snapshots whose quarter ended on/before `asof` (expanding window). Counts, not dollars.
    """
    min_q = MIN_TRAIN_QUARTERS if min_train_quarters is None else min_train_quarters
    k_cell = SHRINK_K_CELL if k_cell is None else k_cell
    k_stage = SHRINK_K_STAGE if k_stage is None else k_stage
    t = labeled[labeled["qend"] <= pd.Timestamp(asof)]
    nq = int(t["qlabel"].nunique())
    if nq < min_q or len(t) == 0:
        return {"ok": False, "n_quarters": nq}
    g = t["label"].mean()
    st = t.groupby("key")["label"].agg(["sum", "count"])
    stage_p = {k: (r["sum"] + k_stage * g) / (r["count"] + k_stage) for k, r in st.iterrows()}
    ce = t.groupby(["key", "bucket", "product"])["label"].agg(["sum", "count"])
    cell_p = {k: (r["sum"] + k_cell * stage_p[k[0]]) / (r["count"] + k_cell) for k, r in ce.iterrows()}
    return {"ok": True, "n_quarters": nq, "global": g, "stage": stage_p, "cell": cell_p}


def p_win(rates, key, bucket, product):
    """Fallback chain: cell -> stage -> global."""
    return rates["cell"].get((key, bucket, product), rates["stage"].get(key, rates["global"]))


def amount_scale(prepared, arr_df, product, asof):
    """History `amount` -> Growth ARR multiplier = sum(realised ARR) / sum(history amount at win time).

    ASSUMPTION: OpportunityHistory.amount is opportunity-level (e.g. TCV/total) while Growth ARR is
    amount_towards_plan on opp-products, so the two differ by a roughly stable ratio per product.
    Learned only from opps whose state AS OF `asof` is Won with close_date before `asof`.
    Falls back to 1.0 (and says so via the second return value) if there is no data.
    """
    asof = pd.Timestamp(asof)
    w = prepared[(prepared["asof"] == asof) & (prepared["product"] == product)
                 & (prepared["key"] == "won") & (prepared["close_date"] < asof)]
    w = w.merge(arr_df[arr_df["product"] == product][["opp_id", "arr"]], on="opp_id", how="inner")
    amt = w["amount"].sum()
    if len(w) == 0 or amt <= 0:
        return 1.0, "no_resolved_deals_scale=1.0"
    sc = float(np.clip(w["arr"].sum() / amt, MIN_AMOUNT_SCALE, MAX_AMOUNT_SCALE))
    return sc, f"scale_from_{len(w)}_won_opps"


def nowcast(open_rows, qtd_actual, rates, scale, product):
    """nowcast = QTD actual + sum(amount * scale * p_win) over open in-quarter opps. Returns dict.

    coverage_ratio = unweighted scaled open pipeline / weighted open pipeline (x; higher = more slack).
    """
    r = open_rows.copy()
    r["p"] = [p_win(rates, k, b, product) for k, b in zip(r["key"], r["bucket"])]
    r["arr_est"] = r["amount"] * scale
    r["w_arr"] = r["arr_est"] * r["p"]
    by_stage = r.groupby("key")["w_arr"].sum().to_dict()
    wsum = float(r["w_arr"].sum())
    usum = float(r["arr_est"].sum())
    return {"qtd_actual": float(qtd_actual), "weighted_open_pipeline": wsum,
            "unweighted_open_pipeline": usum, "nowcast_landing": float(qtd_actual) + wsum,
            "coverage_ratio": (usum / wsum) if wsum > 0 else float("nan"),
            "by_stage": by_stage, "n_open": int(len(r))}

In [ ]:
# SELF-TEST (no Spark): leakage traps + hand-computed arithmetic. Fails loudly on any violation.
def _self_test():
    T = pd.Timestamp
    asof = T("2025-02-03")                      # a Monday
    hist = pd.DataFrame([
        # A: Qualify, then Won EXACTLY at as-of (must be ignored), then Won later (ignored)
        dict(id="1", opp_id="A", change_ts=T("2025-01-10"), stage="2. Qualify", forecast_cat="Pipeline", amount=100, close_date=T("2025-03-15")),
        dict(id="2", opp_id="A", change_ts=asof, stage="5. Won", forecast_cat="Closed", amount=100, close_date=T("2025-02-03")),
        dict(id="3", opp_id="A", change_ts=T("2025-02-20"), stage="5. Won", forecast_cat="Closed", amount=120, close_date=T("2025-02-20")),
        # B: two rows before as-of; the later one (amount 300) must win
        dict(id="4", opp_id="B", change_ts=T("2025-01-05"), stage="1. Discovery", forecast_cat="Pipeline", amount=200, close_date=T("2025-03-20")),
        dict(id="5", opp_id="B", change_ts=T("2025-01-25"), stage="4. Negotiate", forecast_cat="Commit", amount=300, close_date=T("2025-03-25")),
        # C: first appears after as-of -> must not exist in the state
        dict(id="6", opp_id="C", change_ts=T("2025-02-10"), stage="2. Qualify", forecast_cat="Pipeline", amount=50, close_date=T("2025-03-30")),
    ])
    st = asof_state(hist, asof).set_index("opp_id")
    assert set(st.index) == {"A", "B"}, f"asof_state leaked or dropped opps: {set(st.index)}"
    assert st.loc["A", "stage"] == "2. Qualify" and st.loc["A", "amount"] == 100, "row at/after as-of leaked"
    assert st.loc["B", "amount"] == 300 and st.loc["B", "stage"] == "4. Negotiate", "latest pre-as-of row not chosen"
    assert asof_state(hist, T("2025-02-03 00:00:01")).set_index("opp_id").loc["A", "stage"] == "5. Won"

    # (i-b) trap: A becomes Won after as-of -> snapshot is OPEN (feature), label is 1 (future, label only)
    snaps = st.reset_index().assign(asof=asof, product="UCC")[
        ["asof", "opp_id", "product", "stage", "forecast_cat", "amount", "close_date"]]
    op = open_in_quarter(prepare_snaps(snaps))
    assert set(op["opp_id"]) == {"A", "B"} and op["is_open"].all()
    finals = pd.DataFrame([dict(opp_id="A", final_stage="5. Won", final_forecast_cat="Closed", final_close_date=T("2025-02-20")),
                           dict(opp_id="B", final_stage="Closed Lost", final_forecast_cat="Omitted", final_close_date=T("2025-03-25"))])
    lab = add_labels(op, finals).set_index("opp_id")["label"]
    assert lab["A"] == 1 and lab["B"] == 0, f"labels wrong: {lab.to_dict()}"

    # (ii) win rates only use quarters that ENDED before the as-of being scored
    rows = []
    for qs in ["2024-01-01", "2024-04-01", "2024-07-01", "2024-10-01"]:
        rows.append(dict(key="qualify", bucket=3, product="UCC", qlabel=quarter_label(qs), qend=next_quarter_start(qs), label=0))
    rows.append(dict(key="qualify", bucket=3, product="UCC", qlabel="2025Q1", qend=T("2025-04-01"), label=1))
    lbl = pd.DataFrame(rows)
    r1 = win_rates(lbl, T("2025-01-06"))
    assert r1["ok"] and r1["n_quarters"] == 4 and r1["global"] == 0.0, f"future quarter leaked into training: {r1}"
    r2 = win_rates(lbl, T("2025-04-07"))
    assert r2["n_quarters"] == 5 and r2["global"] > 0, "completed quarter not used"
    assert not win_rates(lbl, T("2024-10-07"))["ok"], "must refuse to train with < MIN_TRAIN_QUARTERS"
    assert 0 < r2["stage"]["qualify"] < 1 and r2["cell"][("qualify", 3, "UCC")] < 0.5   # shrunk, not raw 0.2 / 0

    # (iii) hand-computed nowcast: 1000 + 100*0.5*0.2 + 200*0.5*0.6 = 1070
    rates = {"ok": True, "global": 0.1, "stage": {"qualify": 0.2, "negotiate": 0.6}, "cell": {}}
    o = pd.DataFrame([dict(key="qualify", bucket=2, amount=100.0), dict(key="negotiate", bucket=2, amount=200.0)])
    nc = nowcast(o, 1000.0, rates, 0.5, "UCC")
    assert abs(nc["weighted_open_pipeline"] - 70.0) < 1e-9 and abs(nc["nowcast_landing"] - 1070.0) < 1e-9
    assert abs(nc["coverage_ratio"] - 150.0 / 70.0) < 1e-9 and abs(nc["by_stage"]["negotiate"] - 60.0) < 1e-9
    # scale learned from deals resolved BEFORE as-of only
    sn = prepare_snaps(pd.DataFrame([
        dict(asof=asof, opp_id="W1", product="UCC", stage="Closed Won", forecast_cat="Closed", amount=200, close_date=T("2025-01-20")),
        dict(asof=asof, opp_id="W2", product="UCC", stage="Closed Won", forecast_cat="Closed", amount=999, close_date=T("2025-02-03"))]))
    arr = pd.DataFrame([dict(opp_id="W1", product="UCC", arr=100.0), dict(opp_id="W2", product="UCC", arr=1.0)])
    assert abs(amount_scale(sn, arr, "UCC", asof)[0] - 0.5) < 1e-9, "scale used a deal not resolved before as-of"

    # (iv) quarter / bucket maths incl. year boundary
    assert quarter_start("2024-12-30") == T("2024-10-01") and next_quarter_start("2024-12-30") == T("2025-01-01")
    assert quarter_label("2025-01-01") == "2025Q1" and quarter_label("2024-12-31") == "2024Q4"
    assert weeks_to_qend(T("2024-12-23")) == 1 and weeks_bucket(1) == 0 and weeks_bucket(2) == 1 and weeks_bucket(11) == 5
    assert asof_monday_for(T("2025-01-01"), 1) == T("2025-01-06") and asof_monday_for(T("2025-01-01"), 5) == T("2025-02-03")
    assert asof_monday_for(T("2025-01-01"), 1).weekday() == 0
    assert stage_key("7. Won") == "won" and stage_key("Closed Lost") == "lost" and stage_key("3. Evaluate") == "evaluate"
    assert stage_key(None, "Commit") == "commit" and not is_open_key("won") and is_open_key("qualify")
    print("SELF-TEST PASSED (asof_state, labels, win-rate window, nowcast arithmetic, quarter maths)")


if RUN_SELF_TEST:
    _self_test()

In [ ]:
# DISCOVERY + SCHEMA VALIDATION of the (UNVERIFIED) OpportunityHistory table.
from pyspark.sql import functions as F, Window

print("Candidate history tables (information_schema, per catalog):")
for _cat in ["datagroup", "datagroup_mdl"]:
    try:
        _c = spark.sql(f"""SELECT table_catalog, table_schema, table_name FROM {_cat}.information_schema.tables
                           WHERE LOWER(table_name) LIKE '%opportunity_history%' OR LOWER(table_name) LIKE '%opportunityhistory%'""").collect()
        for r in _c:
            print(f"  {r.table_catalog}.{r.table_schema}.{r.table_name}")
        if not _c:
            print(f"  (none in {_cat})")
    except Exception as e:
        print(f"  could not search {_cat}: {str(e)[:120]}")
print(f"Using HISTORY_TABLE = {HISTORY_TABLE}  (override the constant in the config cell if wrong)\n")

try:
    _hist_raw = spark.table(HISTORY_TABLE)
except Exception as e:
    raise RuntimeError(f"Cannot read HISTORY_TABLE={HISTORY_TABLE}: {e}. Pick one of the candidates above.")

_actual_cols = _hist_raw.columns
_lower = {c.lower(): c for c in _actual_cols}
RESOLVED = {}
for _logical, _cands in COLUMN_MAP.items():
    _hit = next((_lower[c.lower()] for c in _cands if c.lower() in _lower), None)
    if _hit:
        RESOLVED[_logical] = _hit
_missing = [k for k in REQUIRED_LOGICAL if k not in RESOLVED]
if _missing:
    raise RuntimeError(f"History table lacks required columns {_missing}. Actual columns: {_actual_cols}. "
                       f"Edit COLUMN_MAP in the config cell.")
print("Resolved columns:", RESOLVED)

_c = RESOLVED
_latest_day = _hist_raw.agg(F.max(F.col(_c["data_day"]).cast("string"))).first()[0]
_h = _hist_raw.filter(F.col(_c["data_day"]).cast("string") == _latest_day)
_h = _h.withColumn("_ts", F.to_timestamp(F.col(_c["change_ts"])))
_s = _h.agg(F.count("*").alias("rows"), F.countDistinct(_c["id"]).alias("distinct_ids"),
            F.countDistinct(_c["opp_id"]).alias("distinct_opps"),
            F.min("_ts").alias("min_ts"), F.max("_ts").alias("max_ts")).first()
print(f"latest data_day={_latest_day} rows={_s.rows:,} distinct_ids={_s.distinct_ids:,} "
      f"distinct_opps={_s.distinct_opps:,} change_ts range={_s.min_ts} .. {_s.max_ts}")
assert _s.min_ts is not None, f"change timestamp column {_c['change_ts']} could not be parsed to a timestamp."
_years = (_s.max_ts - _s.min_ts).days / 365.25
assert _years >= 3, f"Only {_years:.1f} years of history (need >= 3): not enough for an expanding-window backtest."

# Point-in-time check: opps whose latest row is Won must show >1 distinct stage over time.
_stage = _c["stage"]
_won_opps = (_h.filter(F.lower(F.col(_stage)).contains("won")).select(F.col(_c["opp_id"]).alias("o")).distinct().limit(2000))
_sample = _h.join(_won_opps, _h[_c["opp_id"]] == _won_opps["o"]).groupBy(_c["opp_id"]).agg(F.countDistinct(_stage).alias("n"))
_ps = _sample.agg(F.count("*").alias("n_opps"), F.avg((F.col("n") > 1).cast("double")).alias("share_multi")).first()
print(f"won-opp sample: {_ps.n_opps} opps, share with >1 distinct stagename over time = {_ps.share_multi}")
if not _ps.n_opps or _ps.share_multi < 0.5:
    raise RuntimeError("history is not point-in-time: won opps do not show stage progression "
                       f"(share_multi={_ps.share_multi}). Do NOT use this table for the nowcast.")
print("Stage values:", [r[0] for r in _h.groupBy(_stage).count().orderBy(F.desc("count")).limit(25).collect()])
print("Schema validation PASSED")

In [ ]:
# LOAD: clean history (latest data_day, dedupe by id, drop deleted), opp classification, actuals.
_c = RESOLVED
_ch = "'" + "','".join(ITSG_EXCLUDED_CHANNELS) + "'"

# --- history: one row per history id, normalised column names ----------------
_h = (spark.table(HISTORY_TABLE)
      .filter(F.col(_c["data_day"]).cast("string") == _latest_day))          # latest daily dump only
if "is_deleted" in _c:
    _h = _h.filter(~F.coalesce(F.col(_c["is_deleted"]).cast("boolean"), F.lit(False)))
_fc = F.col(_c["forecast_cat"]).cast("string") if "forecast_cat" in _c else F.lit(None).cast("string")
hist_sdf = (_h.select(F.col(_c["id"]).cast("string").alias("id"),
                      F.col(_c["opp_id"]).cast("string").alias("opp_id"),
                      F.to_timestamp(F.col(_c["change_ts"])).alias("change_ts"),
                      F.col(_c["stage"]).cast("string").alias("stage"),
                      _fc.alias("forecast_cat"),
                      F.col(_c["amount"]).cast("double").alias("amount"),
                      F.to_date(F.col(_c["close_date"])).alias("close_date"))
            .filter(F.col("change_ts").isNotNull())
            .dropDuplicates(["id"]))                                          # dedupe by history id
print("history rows after clean:", hist_sdf.count())

# --- classification: one product per opportunity (largest in-scope Growth ARR; ties -> name) -----
# Scope per product mirrors notebooks 01 (UCC: demo_stage=0) and 02 (ITSG: channel exclusion). NO is_won filter,
# so open opportunities are classified too. ASSUMPTION: actual_opportunity_id is the Opportunity Id.
_oli = f"""
SELECT actual_opportunity_id AS opp_id, UPPER(product_group) AS product,
       CAST(amount_towards_plan AS DOUBLE) AS atp
FROM {BASE_TABLE}
WHERE actual_opportunity_id IS NOT NULL AND UPPER(COALESCE(purchase_type_rollup,'')) = 'GROWTH'
  AND (  (UPPER(COALESCE(product_group,'')) = 'UCC' AND demo_stage = 0)
      OR (UPPER(COALESCE(product_group,'')) = 'ITSG' AND COALESCE(sales_channel,'') NOT IN ({_ch})) )
"""
_cls = spark.sql(_oli).groupBy("opp_id", "product").agg(F.sum(F.coalesce("atp", F.lit(0.0))).alias("atp_sum"))
_w = Window.partitionBy("opp_id").orderBy(F.desc("atp_sum"), F.asc("product"))
opp_class_sdf = _cls.withColumn("_r", F.row_number().over(_w)).filter("_r = 1").select("opp_id", "product")
print("classified opps by product:", {r["product"]: r["count"] for r in opp_class_sdf.groupBy("product").count().collect()})

# --- ACTUALS: exactly the foundation filters (won+closed, GROWTH, close_date >= START_DATE) ----
# Daily grain so QTD can be cut at any Monday; weeks (Monday) are DATE_TRUNC('week') as in 01/02.
_act_sql = lambda prod, extra: f"""
SELECT '{prod}' AS product, CAST(close_date AS DATE) AS close_date, actual_opportunity_id AS opp_id,
       CAST(amount_towards_plan AS DOUBLE) AS arr
FROM {BASE_TABLE}
WHERE is_won = true AND is_closed = true AND close_date >= DATE '{START_DATE}' AND close_date <= CURRENT_DATE()
  AND UPPER(COALESCE(product_group,'')) = '{prod}' AND UPPER(COALESCE(purchase_type_rollup,'')) = 'GROWTH' {extra}"""
actual_pdf = pd.concat([
    spark.sql(_act_sql("UCC", "AND demo_stage = 0")).toPandas(),
    spark.sql(_act_sql("ITSG", f"AND COALESCE(sales_channel,'') NOT IN ({_ch})")).toPandas()], ignore_index=True)
actual_pdf["close_date"] = pd.to_datetime(actual_pdf["close_date"])
actual_pdf["arr"] = pd.to_numeric(actual_pdf["arr"], errors="coerce").fillna(0.0)
actual_pdf["week_start"] = actual_pdf["close_date"].map(week_monday)
print(actual_pdf.groupby("product")["arr"].agg(["count", "sum"]))
assert (actual_pdf.groupby("product")["arr"].sum() > 0).all(), "an actuals series is empty: check filters/BASE_TABLE"

# realised ARR per opp for the amount scale (only opps whose classified product matches)
_cls_pdf = opp_class_sdf.toPandas()
arr_df = (actual_pdf.dropna(subset=["opp_id"]).groupby(["opp_id", "product"], as_index=False)["arr"].sum()
          .merge(_cls_pdf, on=["opp_id", "product"], how="inner"))
print("opps with realised ARR usable for scale:", len(arr_df))

# coverage check: history opps that classify into UCC/ITSG
_cov = hist_sdf.select("opp_id").distinct().join(opp_class_sdf, "opp_id")
print("distinct history opps matched to a UCC/ITSG classification:", _cov.count())

In [ ]:
# SPARK AS-OF RECONSTRUCTION (set-based, no per-opp Python loops).
# For every needed as-of Monday M: per opportunity, the history row with the greatest (change_ts, id) such that
# change_ts < M (STRICT). Implemented as join + groupBy(max(struct)); same semantics as asof_state().
_today_monday = week_monday(pd.Timestamp(date.today()))
_hmin = hist_sdf.agg(F.min("change_ts")).first()[0]
_q = quarter_start(_hmin) + pd.DateOffset(months=3)       # first quarter must start after history begins
ASOF_DATES = []                                            # (asof Monday, quarter_label, week_no)
while _q <= _today_monday:
    for _w in ASOF_WEEKS:
        _m = asof_monday_for(_q, _w)
        if _m <= _today_monday and quarter_start(_m) == _q:
            ASOF_DATES.append((_m, quarter_label(_q), _w))
    _q = _q + pd.DateOffset(months=3)
_cur_asof = _today_monday
if _cur_asof not in [a for a, _, _ in ASOF_DATES]:
    ASOF_DATES.append((_cur_asof, quarter_label(_cur_asof), None))   # live scoring date (not a backtest point)
print(f"{len(ASOF_DATES)} as-of Mondays, {ASOF_DATES[0][0].date()} .. {ASOF_DATES[-1][0].date()}")

asof_sdf = spark.createDataFrame(pd.DataFrame({"asof": [a for a, _, _ in ASOF_DATES]})
                                 ).withColumn("asof", F.col("asof").cast("timestamp"))
_hc = hist_sdf.join(opp_class_sdf, "opp_id")                 # only UCC/ITSG Growth opps
_st = F.struct("change_ts", "id", "stage", "forecast_cat", "amount", "close_date")
_snap = (_hc.join(asof_sdf, _hc["change_ts"] < asof_sdf["asof"])            # STRICTLY before the as-of Monday
            .groupBy("opp_id", "product", "asof").agg(F.max(_st).alias("s"))
            .select("opp_id", "product", "asof", "s.stage", "s.forecast_cat", "s.amount", "s.close_date"))
# Keep only what the model uses: close date within [as-of - 12 months, end of as-of quarter) -> open-in-quarter
# deals plus trailing-12m won deals (amount-scale training). Filtering uses as-of-state values only.
_snap = _snap.filter((F.col("close_date") >= F.add_months(F.col("asof").cast("date"), -12)) &
                     (F.col("close_date") < F.add_months(F.date_trunc("quarter", F.col("asof")).cast("date"), 3)))
snaps_pdf = _snap.toPandas()
snaps_pdf["asof"] = pd.to_datetime(snaps_pdf["asof"])

# FINAL state per opp: used ONLY by add_labels() (labels may look at the future; features never do).
_fin = (_hc.groupBy("opp_id").agg(F.max(_st).alias("s"))
           .select("opp_id", F.col("s.stage").alias("final_stage"), F.col("s.forecast_cat").alias("final_forecast_cat"),
                   F.col("s.close_date").alias("final_close_date")))
finals_pdf = _fin.join(_snap.select("opp_id").distinct(), "opp_id").toPandas()
print(f"snapshot rows: {len(snaps_pdf):,}; opps with finals: {len(finals_pdf):,}")
print("stage keys seen:", pd.Series([stage_key(a, b) for a, b in zip(snaps_pdf['stage'], snaps_pdf['forecast_cat'])]).value_counts().to_dict())

# sanity: unmapped stage names fall into other_open -- surface them
_unk = snaps_pdf[[stage_key(a, b) == "other_open" for a, b in zip(snaps_pdf["stage"], snaps_pdf["forecast_cat"])]]["stage"].value_counts().head(10)
if len(_unk):
    print("WARNING: stage names mapped to 'other_open' (extend stage_key if material):", _unk.to_dict())

In [ ]:
# BACKTEST HELPERS (pure python): actual landings, baselines, blend, gate. Every parameter uses PRIOR quarters only.
def qtd_actual_fn(act, product, qstart, asof):
    a = act[(act["product"] == product) & (act["close_date"] >= qstart) & (act["close_date"] < asof)]
    return float(a["arr"].sum())


def landing_fn(act, product, qstart):
    a = act[(act["product"] == product) & (act["close_date"] >= qstart) & (act["close_date"] < next_quarter_start(qstart))]
    return float(a["arr"].sum())


def completed_quarters(asof, n=12):
    """Quarter starts that END on/before asof, most recent first."""
    qs, q = [], quarter_start(asof)
    for _ in range(n):
        q = q - pd.DateOffset(months=3)
        qs.append(q)
    return qs


def baseline_seasonal(act, product, asof):
    """(a) landing of the same quarter last year x (last 4 completed quarters / the 4 quarters a year earlier)."""
    q0 = quarter_start(asof)
    last4 = [q0 - pd.DateOffset(months=3 * i) for i in (1, 2, 3, 4)]
    prev4 = [q - pd.DateOffset(years=1) for q in last4]
    if min(prev4) < pd.Timestamp(START_DATE):
        return float("nan")
    num = sum(landing_fn(act, product, q) for q in last4)
    den = sum(landing_fn(act, product, q) for q in prev4)
    ly = landing_fn(act, product, q0 - pd.DateOffset(years=1))
    return float(ly * num / den) if den > 0 else float("nan")


def baseline_pace(act, product, asof, week_no, min_q=None):
    """(b) QTD / share, share = prior quarters' (QTD at same week) / landing, ratio of sums. Prior quarters only."""
    min_q = MIN_TRAIN_QUARTERS if min_q is None else min_q
    qd = qtd_actual_fn(act, product, quarter_start(asof), asof)
    num = den = 0.0
    n = 0
    for q in completed_quarters(asof):
        if q < pd.Timestamp(START_DATE):
            continue
        land = landing_fn(act, product, q)
        if land > 0:
            num += qtd_actual_fn(act, product, q, asof_monday_for(q, week_no))
            den += land
            n += 1
    return float(qd / (num / den)) if n >= min_q and num > 0 else float("nan")


def blend_weight(prior, grid=None):
    """(c) weight on the nowcast chosen to minimise MAPE of w*nowcast+(1-w)*seasonal on PRIOR quarters only."""
    grid = np.linspace(0, 1, 11) if grid is None else grid
    p = prior.dropna(subset=["nowcast", "base_seasonal", "actual"])
    p = p[p["actual"] > 0]
    if len(p) < 2:
        return 0.5
    errs = [np.mean(np.abs((w * p["nowcast"] + (1 - w) * p["base_seasonal"] - p["actual"]) / p["actual"])) for w in grid]
    return float(grid[int(np.argmin(errs))])


def mape(pred, actual):
    d = pd.DataFrame({"p": pred, "a": actual}).dropna()
    d = d[d["a"] > 0]
    return float(np.mean(np.abs((d["p"] - d["a"]) / d["a"]))) if len(d) else float("nan")


def gate_result(summary):
    """GATE: nowcast MAPE < best baseline MAPE (a, b, c) at every as-of week >= GATE_MIN_ASOF_WEEK, BOTH products."""
    ok, detail = True, []
    for prod in PRODUCTS:
        for w in [x for x in ASOF_WEEKS if x >= GATE_MIN_ASOF_WEEK]:
            r = summary[(summary["product"] == prod) & (summary["week_no"] == w)]
            if r.empty or r[["mape_nowcast", "mape_seasonal", "mape_pace", "mape_blend"]].isna().any(axis=None):
                ok = False; detail.append(f"{prod} wk{w}: insufficient data"); continue
            r = r.iloc[0]
            best = min(r["mape_seasonal"], r["mape_pace"], r["mape_blend"])
            passed = r["mape_nowcast"] < best
            ok = ok and passed
            detail.append(f"{prod} wk{w}: nowcast {r['mape_nowcast']:.1%} vs best baseline {best:.1%} -> {'PASS' if passed else 'FAIL'}")
    return ok, detail

In [ ]:
# BACKTEST ENGINE (pure python; takes the frames built by the Spark cells).
def score_one(prepared_open, labeled, arr_df, prepared_all, act, product, asof, week_no):
    """Nowcast for one (product, as-of). Win rates / scale use only data resolved BEFORE as-of. None if untrainable."""
    rates = win_rates(labeled, asof)
    if not rates["ok"]:
        return None
    sc, sc_note = amount_scale(prepared_all, arr_df, product, asof)
    rows = prepared_open[(prepared_open["asof"] == asof) & (prepared_open["product"] == product)]
    qd = qtd_actual_fn(act, product, quarter_start(asof), asof)
    nc = nowcast(rows, qd, rates, sc, product)
    nc.update(n_train_quarters=rates["n_quarters"], scale=sc, scale_note=sc_note)
    return nc


def run_backtest(snaps, finals, act, arr_df, asof_dates, today):
    prepared_all = prepare_snaps(snaps)
    prepared_open = open_in_quarter(prepared_all)
    labeled = add_labels(prepared_open, finals)
    out = []
    for asof, qlab, w in asof_dates:
        if w is None or next_quarter_start(asof) > pd.Timestamp(today):
            continue                                   # only quarters with a known landing are backtested
        qs = quarter_start(asof)
        for prod in PRODUCTS:
            nc = score_one(prepared_open, labeled, arr_df, prepared_all, act, prod, asof, w)
            if nc is None:
                continue
            out.append(dict(product=prod, quarter_label=qlab, asof_date=asof, week_no=w,
                            qtd_actual=nc["qtd_actual"], weighted_open_pipeline=nc["weighted_open_pipeline"],
                            nowcast=nc["nowcast_landing"], coverage_ratio=nc["coverage_ratio"],
                            actual=landing_fn(act, prod, qs),
                            base_seasonal=baseline_seasonal(act, prod, asof),
                            base_pace=baseline_pace(act, prod, asof, w),
                            n_train_quarters=nc["n_train_quarters"], scale=nc["scale"]))
    bt = pd.DataFrame(out)
    if bt.empty:
        return bt, pd.DataFrame()
    # (c) blend: weight per (product, week) from prior quarters only (quarter ended before this as-of)
    bt["blend_weight"], bt["base_blend"] = np.nan, np.nan
    for i, r in bt.iterrows():
        prior = bt[(bt["product"] == r["product"]) & (bt["week_no"] == r["week_no"])
                   & (bt["quarter_label"] != r["quarter_label"])
                   & (bt["asof_date"].map(next_quarter_start) <= r["asof_date"])]
        wgt = blend_weight(prior)
        bt.at[i, "blend_weight"] = wgt
        bt.at[i, "base_blend"] = wgt * r["nowcast"] + (1 - wgt) * r["base_seasonal"]
    # Total = UCC + ITSG (only where both exist)
    piv = bt.pivot_table(index=["quarter_label", "asof_date", "week_no"], columns="product",
                         values=["qtd_actual", "weighted_open_pipeline", "nowcast", "actual", "base_seasonal",
                                 "base_pace", "base_blend"], aggfunc="sum")
    tot = pd.DataFrame({k: piv[k][PRODUCTS].sum(axis=1, min_count=2) for k in piv.columns.levels[0]}).reset_index()
    tot["product"] = "Total"
    bt = pd.concat([bt, tot], ignore_index=True)
    bt["err_pct"] = (bt["nowcast"] - bt["actual"]) / bt["actual"]          # signed: + = over-forecast
    rows = []
    for (prod, w), g in bt.groupby(["product", "week_no"]):
        # like-for-like: score every method on the SAME quarters (those where all four predictions exist)
        g = g.dropna(subset=["nowcast", "base_seasonal", "base_pace", "base_blend", "actual"])
        rows.append(dict(product=prod, week_no=int(w), n_quarters=int(len(g)),
                         mape_nowcast=mape(g["nowcast"], g["actual"]), mape_seasonal=mape(g["base_seasonal"], g["actual"]),
                         mape_pace=mape(g["base_pace"], g["actual"]), mape_blend=mape(g["base_blend"], g["actual"]),
                         mean_signed_err=float(g["err_pct"].mean())))
    return bt, pd.DataFrame(rows)

In [ ]:
# RUN BACKTEST + GATE
bt_pdf, bt_summary = run_backtest(snaps_pdf, finals_pdf, actual_pdf, arr_df, ASOF_DATES, date.today())
if bt_pdf.empty:
    raise RuntimeError("Backtest produced no rows: fewer than MIN_TRAIN_QUARTERS completed quarters of history "
                       f"({MIN_TRAIN_QUARTERS}) before any scored quarter, or no UCC/ITSG opps matched. See load cells.")

pd.set_option("display.width", 200, "display.max_columns", 30)
print("MAPE by product x as-of week (like-for-like quarters):")
print(bt_summary.sort_values(["product", "week_no"]).round(3).to_string(index=False))

GATE_OK, GATE_DETAIL = gate_result(bt_summary)
print("\nGATE RULE: nowcast MAPE < best baseline (seasonal / pace / blend) at as-of weeks >= "
      f"{GATE_MIN_ASOF_WEEK} for BOTH UCC and ITSG.")
for _d in GATE_DETAIL:
    print("  ", _d)
print(f"GATE (validated) = {GATE_OK}")
if not GATE_OK:
    print("Nowcast is NOT validated: rows are written with validated=False; do not add this task to the weekly job.")

In [ ]:
# SCORE THE CURRENT QUARTER at the current as-of Monday (point-in-time state, win rates from completed quarters only).
RUN_DATE = date.today()
_asof = _cur_asof
_qlab = quarter_label(_asof)
_wk_now = int((_asof - first_monday_on_or_after(quarter_start(_asof))).days // 7) + 1
_wk_match = min(ASOF_WEEKS, key=lambda x: abs(x - _wk_now))      # nearest backtested as-of week for residual quantiles
print(f"as-of {_asof.date()} = week {_wk_now} of {_qlab}; residual quantiles from backtest week {_wk_match}")

_prep_all = prepare_snaps(snaps_pdf)
_prep_open = open_in_quarter(_prep_all)
_labeled = add_labels(_prep_open, finals_pdf)


def _interval(product, nowcast_value):
    """Empirical p25/p75 of backtest residual ratios (actual-nowcast)/nowcast at the matching as-of week."""
    g = bt_pdf[(bt_pdf["product"] == product) & (bt_pdf["week_no"] == _wk_match)].dropna(subset=["nowcast", "actual"])
    g = g[g["nowcast"] > 0]
    if len(g) < 3:
        return float("nan"), float("nan"), f"quantiles unavailable (n={len(g)}<3 backtest residuals)"
    r = ((g["actual"] - g["nowcast"]) / g["nowcast"]).to_numpy()
    q25, q75 = np.quantile(r, [0.25, 0.75])
    return nowcast_value * (1 + q25), nowcast_value * (1 + q75), f"empirical residual quantiles n={len(g)}"


_res = {}
for _p in PRODUCTS:
    _res[_p] = score_one(_prep_open, _labeled, arr_df, _prep_all, actual_pdf, _p, _asof, _wk_now)
    if _res[_p] is None:
        raise RuntimeError(f"Cannot score {_p}: fewer than {MIN_TRAIN_QUARTERS} completed training quarters.")

_rows = []
for _p in PRODUCTS + ["Total"]:
    if _p == "Total":
        _qtd = sum(_res[x]["qtd_actual"] for x in PRODUCTS)
        _w = sum(_res[x]["weighted_open_pipeline"] for x in PRODUCTS)
        _u = sum(_res[x]["unweighted_open_pipeline"] for x in PRODUCTS)
        _cov = _u / _w if _w > 0 else float("nan")
        _ntq = min(_res[x]["n_train_quarters"] for x in PRODUCTS)
        _note = "Total = UCC + ITSG"
    else:
        r = _res[_p]
        _qtd, _w, _cov, _ntq = r["qtd_actual"], r["weighted_open_pipeline"], r["coverage_ratio"], r["n_train_quarters"]
        _note = f"{r['scale_note']}; scale={r['scale']:.3f}; n_open={r['n_open']}; stages={ {k: round(v) for k, v in r['by_stage'].items()} }"
    _nc = _qtd + _w
    _p25, _p75, _qnote = _interval(_p, _nc)
    _rows.append(dict(run_date=RUN_DATE, asof_date=_asof.date(), product=_p, quarter_label=_qlab,
                      qtd_actual=_qtd, weighted_open_pipeline=_w, nowcast_landing=_nc,
                      nowcast_p25=_p25, nowcast_p75=_p75, coverage_ratio=_cov,
                      method="stage_weighted_open_pipeline_v1", validated=bool(GATE_OK),
                      n_train_quarters=int(_ntq), notes=f"{_note}; {_qnote}; gate={GATE_OK}"))
pipeline_pdf = pd.DataFrame(_rows)
print(pipeline_pdf.drop(columns=["notes"]).round(2).to_string(index=False))

In [ ]:
# WRITE (own tables only; nothing existing is touched). Write and GRANT are reported separately (as notebook 03).
_bt_out = bt_pdf.copy()
_bt_out.insert(0, "run_date", RUN_DATE)
_bt_out["asof_date"] = pd.to_datetime(_bt_out["asof_date"]).dt.date
_bt_out["validated"] = bool(GATE_OK)

_targets = [(PIPELINE_TABLE, pipeline_pdf), (BACKTEST_TABLE, _bt_out)]
for _tbl, _df in _targets:
    try:
        spark.createDataFrame(_df).write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(_tbl)
        print(f"✓ Wrote {_tbl} ({len(_df)} rows, run_date={RUN_DATE})")
    except Exception as e:
        print(f"✗ WRITE FAILED for {_tbl}: {e}")
        raise

# Grants are non-fatal (e.g. missing MANAGE); reported separately from the writes above.
for _tbl, _ in _targets:
    try:
        spark.sql(f"GRANT SELECT ON TABLE {_tbl} TO `{APP_SERVICE_PRINCIPAL}`")
        print(f"✓ Granted SELECT on {_tbl}")
    except Exception as e:
        print(f"⚠ GRANT failed on {_tbl} (table WAS written): {str(e)[:200]}")

print(f"\nDone. validated={GATE_OK}. {'Safe to consider adding to the weekly job.' if GATE_OK else 'Keep OUT of the weekly job.'}")